# Step 20: Advanced: Human Approval and Checkpointed Workflows

        _Learner Notebook_  
        Level: **Advanced**  
        Estimated time: **105 minutes**

        ## Learning Objectives
        - [ ] Design approval boundaries for risky actions.
- [ ] Persist enough workflow state to resume reliably.
- [ ] Differentiate audit records from execution payloads.
- [ ] Think through operator-facing workflow recovery paths.

        ## Prerequisites
        - Core Step 14 completed.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Reflection & Next Experiments
8. Summary & Key Takeaways
9. Additional Resources


## Setup & Imports

Supplemental notebooks assume the core helpers are available and that you have already worked
through the matching core lessons. The import cell intentionally keeps the same bootstrap shape
as the core course so you can focus on the deeper pattern rather than environment setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

This advanced notebook treats human oversight as a system design problem: what requires approval, what state must be saved, and how operators re-enter an interrupted flow safely.

This notebook is intentionally deeper than the core path. Expect more design discussion,
more implementation sections, and more open-ended exercises.


## Part 2: Core Implementation

### Define auditable approval records


In [ ]:
from dataclasses import dataclass, asdict
from datetime import datetime

@dataclass
class ApprovalRecord:
    approval_id: str
    action: str
    reason: str
    payload: dict
    requested_at: str
    status: str = "pending"

approvals: dict[str, ApprovalRecord] = {}


## Part 2: Core Implementation

### Create explicit checkpoint snapshots


In [ ]:
checkpoint_store: dict[str, dict] = {}

def save_checkpoint(name: str, payload: dict) -> None:
    checkpoint_store[name] = {
        "saved_at": datetime.utcnow().isoformat(),
        "payload": payload,
    }

def load_checkpoint(name: str) -> dict:
    return checkpoint_store[name]["payload"]


## Part 2: Core Implementation

### Model a gated workflow stage


In [ ]:
def request_approval(action: str, reason: str, payload: dict) -> ApprovalRecord:
    approval_id = f"approval_{len(approvals) + 1}"
    record = ApprovalRecord(
        approval_id=approval_id,
        action=action,
        reason=reason,
        payload=payload,
        requested_at=datetime.utcnow().isoformat(),
    )
    approvals[approval_id] = record
    return record

def approve(approval_id: str) -> None:
    approvals[approval_id].status = "approved"

def reject(approval_id: str) -> None:
    approvals[approval_id].status = "rejected"


## Part 2: Core Implementation

### Run a checkpointed approval flow


In [ ]:
draft_report = {
    "dataset": "quarterly_launch_notes",
    "risk_level": "high",
    "action": "publish_external_summary",
}
save_checkpoint("draft_created", draft_report)

approval = request_approval(
    action="publish_external_summary",
    reason="External publication requires human review.",
    payload=draft_report,
)
approve(approval.approval_id)

print_json(asdict(approval), title="Approval record")
print_json(load_checkpoint("draft_created"), title="Checkpoint payload")


## Part 3: Hands-On Exercises

### Exercise 1
Add a second checkpoint after approval so the resume path can tell whether it should continue to publication or return to review.

### Exercise 2
Write a resume helper that decides the next action based on the latest checkpoint state.

Work through both guided exercises before comparing against the solutions.


In [ ]:
# TODO: save a second checkpoint that represents post-approval state


In [ ]:
def resume_decision(checkpoint_name: str) -> str:
    # TODO: return the next action string
    return ""


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
save_checkpoint(
    "approved_for_publication",
    {"next_stage": "publish", "approval_id": approval.approval_id, "status": approval.status},
)
print_json(load_checkpoint("approved_for_publication"), title="Exercise 1 solution")


In [ ]:
def resume_decision(checkpoint_name: str) -> str:
    payload = load_checkpoint(checkpoint_name)
    if payload.get("next_stage") == "publish":
        return "continue_to_publication"
    return "return_to_review"

print_json(
    {"decision": resume_decision("approved_for_publication")},
    title="Exercise 2 solution",
)


## Part 5: Best Practices & Tips

        - Define approvals around business risk, not around arbitrary complexity.
- Save checkpoints that are meaningful to operators, not just to code.
- Record approval and execution state separately for cleaner audit trails.
- Design resume logic before you need it in an outage or long-running task.


## Reflection & Next Experiments

- Which part of `Advanced: Human Approval and Checkpointed Workflows` felt like the biggest jump from the core course?
- What would you keep deterministic before turning this into a live production feature?
- Where would you add tests, traces, or operator controls before shipping this pattern?


## Summary & Key Takeaways

You deepened the HITL pattern into an auditable, resumable workflow design rather than a single approval prompt.


## Additional Resources

        - `core notebook: 14_human_in_the_loop.ipynb`
- `docs/troubleshooting.md`
